## Example 1: Safe XML Parsing vs Vulnerable Parsing

In [ ]:
import defusedxml.ElementTree as ET
import defusedxml
from xml.etree import ElementTree as StandardET

print("Example 1: Safe XML Parsing vs Vulnerable Parsing")

try:
    print("✓ defusedxml package information:")
    print(f"  Module type: {type(defusedxml).__name__}")
    
    # Show available secure modules
    secure_modules = [attr for attr in dir(defusedxml) if not attr.startswith('_')]
    print(f"  Available secure modules: {secure_modules}")
    
    # Basic safe XML parsing
    print("✓ Safe XML parsing with defusedxml:")
    
    # Safe XML content for demonstration
    safe_xml = """<?xml version="1.0" encoding="UTF-8"?>
    <catalog>
        <book id="1">
            <title>Python Security Guide</title>
            <author>John Doe</author>
            <price currency="USD">29.99</price>
        </book>
        <book id="2">
            <title>Web Application Security</title>
            <author>Jane Smith</author>
            <price currency="EUR">34.50</price>
        </book>
    </catalog>"""
    
    # Parse with defusedxml (secure)
    print("  Parsing safe XML with defusedxml:")
    safe_root = ET.fromstring(safe_xml)
    print(f"    Root tag: {safe_root.tag}")
    print(f"    Number of books: {len(safe_root.findall('book'))}")
    
    for book in safe_root.findall('book'):
        book_id = book.get('id')
        title = book.find('title').text
        author = book.find('author').text
        price = book.find('price')
        print(f"    Book {book_id}: {title} by {author} - {price.text} {price.get('currency')}")
    
    # Demonstrate security features
    print("✓ Security feature comparison:")
    
    print("  defusedxml security features:")
    security_features = [
        "XMLParser with entity restrictions",
        "Protection against billion laughs attack",
        "XXE (XML External Entity) attack prevention",
        "DTD processing restrictions",
        "Entity expansion limits",
        "Recursive entity detection"
    ]
    
    for feature in security_features:
        print(f"    • {feature}")
    
    # Show defusedxml configuration
    print("✓ defusedxml configuration:")
    
    # Check if we can access global configuration
    if hasattr(defusedxml, 'GlobalParserTLS'):
        print("  Global parser configuration available")
    
    # Show parser creation with custom limits
    print("  Creating secure parser with custom limits:")
    try:
        parser = ET.XMLParser()
        print(f"    Parser created: {type(parser).__name__}")
        print(f"    Parser has security restrictions built-in")
    except Exception as e:
        print(f"    Parser creation: {e}")
        raise
    
    # Demonstrate safe parsing of potentially dangerous content
    print("✓ Handling potentially dangerous XML safely:")
    
    # XML with external entities (would be dangerous with standard parser)
    potentially_dangerous_xml = """<?xml version="1.0" encoding="UTF-8"?>
    <!DOCTYPE root [
        <!ENTITY harmless "This is safe content">
    ]>
    <root>
        <data>&harmless;</data>
        <safe_content>Regular XML content</safe_content>
    </root>"""
    
    try:
        print("  Parsing XML with DTD entities using defusedxml:")
        dangerous_root = ET.fromstring(potentially_dangerous_xml)
    except defusedxml.EntitiesForbidden as e:
        # defusedxml forbids entity declarations by default (forbid_entities=True), even a harmless one.
        assert e.name == "harmless", e
        print(f"    ✓ defusedxml blocked potentially dangerous XML: {type(e).__name__}")
        print(f"    Security message: {str(e)[:100]}")
    else:
        raise AssertionError(f"defusedxml parsed an entity-declaring document it must refuse (root {dangerous_root.tag})")
    
    # Show difference from standard library
    print("✓ Comparison with standard xml library:")
    
    comparison_points = [
        "defusedxml: Restricts entity processing by default",
        "xml.etree: Processes all entities without restrictions", 
        "defusedxml: Prevents XML bomb attacks automatically",
        "xml.etree: Vulnerable to XML bomb attacks",
        "defusedxml: Blocks external entity references",
        "xml.etree: Processes external entities if available"
    ]
    
    for point in comparison_points:
        print(f"    • {point}")
    
    print("✓ Safe XML parsing demonstration completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: defusedxml provides secure XML processing")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"defusedxml example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: XML Security Attack Prevention

In [ ]:
import defusedxml.ElementTree as SafeET
import defusedxml

print("Example 2: XML Security Attack Prevention")

try:
    # XML Bomb (Billion Laughs) Attack Prevention
    print("✓ XML Bomb attack prevention:")
    
    # Simulated XML bomb (simplified version for demonstration)
    xml_bomb = """<?xml version="1.0" encoding="UTF-8"?>
    <!DOCTYPE root [
        <!ENTITY a "AAAAAAAAAA">
        <!ENTITY b "&a;&a;&a;&a;&a;&a;&a;&a;&a;&a;">
        <!ENTITY c "&b;&b;&b;&b;&b;&b;&b;&b;&b;&b;">
    ]>
    <root>
        <data>&c;</data>
    </root>"""
    
    print("  Testing XML bomb protection:")
    try:
        # This should be blocked by defusedxml
        bomb_root = SafeET.fromstring(xml_bomb)
    except defusedxml.EntitiesForbidden as e:
        assert e.name == "a", e
        print(f"    ✓ XML bomb successfully blocked: {type(e).__name__}")
        print(f"    Protection message: {str(e)[:80]}...")
    else:
        raise AssertionError(f"XML bomb was NOT blocked (root {bomb_root.tag})")
    
    # XXE (XML External Entity) Attack Prevention
    print("✓ XXE attack prevention:")
    
    # Simulated XXE attack (safe for demonstration)
    xxe_attack = """<?xml version="1.0" encoding="UTF-8"?>
    <!DOCTYPE root [
        <!ENTITY xxe SYSTEM "file:///etc/passwd">
    ]>
    <root>
        <data>&xxe;</data>
    </root>"""
    
    print("  Testing XXE attack protection:")
    try:
        # This should be blocked by defusedxml
        xxe_root = SafeET.fromstring(xxe_attack)
    except defusedxml.EntitiesForbidden as e:
        assert e.name == "xxe", e
        print(f"    ✓ XXE attack successfully blocked: {type(e).__name__}")
        print(f"    Protection message: {str(e)[:80]}...")
    else:
        raise AssertionError(f"XXE attack was NOT blocked (root {xxe_root.tag})")
    
    # DTD processing restriction
    print("✓ DTD processing restrictions:")
    
    # XML with complex DTD
    complex_dtd = """<?xml version="1.0" encoding="UTF-8"?>
    <!DOCTYPE root [
        <!ELEMENT root (item*)>
        <!ELEMENT item (#PCDATA)>
        <!ENTITY normal "Normal entity content">
    ]>
    <root>
        <item>&normal;</item>
        <item>Regular content</item>
    </root>"""
    
    print("  Testing DTD processing restrictions:")
    try:
        dtd_root = SafeET.fromstring(complex_dtd)
    except defusedxml.EntitiesForbidden as e:
        # The DTD itself is allowed (forbid_dtd=False); its <!ENTITY normal ...> declaration is what is refused.
        assert e.name == "normal", e
        print(f"    DTD processing restricted: {type(e).__name__}")
        print(f"    Restriction message: {str(e)[:80]}...")
    else:
        raise AssertionError(f"defusedxml parsed a DTD that declares an entity (root {dtd_root.tag})")
    
    # Recursive entity attack prevention
    print("✓ Recursive entity attack prevention:")
    
    # Recursive entity definition
    recursive_entity = """<?xml version="1.0" encoding="UTF-8"?>
    <!DOCTYPE root [
        <!ENTITY a "&b;">
        <!ENTITY b "&a;">
    ]>
    <root>
        <data>&a;</data>
    </root>"""
    
    print("  Testing recursive entity protection:")
    try:
        recursive_root = SafeET.fromstring(recursive_entity)
    except defusedxml.EntitiesForbidden as e:
        assert e.name == "a", e
        print(f"    ✓ Recursive entity attack blocked: {type(e).__name__}")
        print(f"    Protection message: {str(e)[:80]}...")
    else:
        raise AssertionError(f"Recursive entity attack was NOT blocked (root {recursive_root.tag})")
    
    # Quadratic blowup attack prevention
    print("✓ Quadratic blowup attack prevention:")
    
    # XML designed to cause quadratic blowup
    quadratic_attack = """<?xml version="1.0" encoding="UTF-8"?>
    <!DOCTYPE root [
        <!ENTITY a "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA">
    ]>
    <root>
        <data>&a;&a;&a;&a;&a;&a;&a;&a;&a;&a;&a;&a;&a;&a;&a;&a;</data>
    </root>"""
    
    print("  Testing quadratic blowup protection:")
    try:
        quadratic_root = SafeET.fromstring(quadratic_attack)
    except defusedxml.EntitiesForbidden as e:
        assert e.name == "a", e
        print(f"    ✓ Quadratic blowup attack blocked: {type(e).__name__}")
        print(f"    Protection message: {str(e)[:80]}...")
    else:
        raise AssertionError(f"Quadratic blowup attack was NOT blocked (root {quadratic_root.tag})")
    
    # Security configuration options
    print("✓ Security configuration:")
    
    security_options = [
        "Entity processing disabled by default",
        "External DTD loading blocked",
        "Network access for DTDs disabled", 
        "Entity expansion limits enforced",
        "Parser creation with security defaults",
        "Configurable security levels"
    ]
    
    for option in security_options:
        print(f"    • {option}")
    
    print("✓ XML security attack prevention completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: defusedxml prevents common XML security attacks")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"defusedxml example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Secure XML Module Replacements

In [ ]:
import defusedxml.ElementTree as SafeET
import defusedxml.minidom as SafeMinidom
import defusedxml.sax as SafeSAX
from defusedxml.sax import make_parser
from xml.sax.handler import ContentHandler

print("Example 3: Secure XML Module Replacements")

try:
    # Demonstrate secure module replacements
    print("✓ Secure XML module replacements:")
    
    # 1. ElementTree replacement
    print("  1. ElementTree secure replacement:")
    xml_data = """<?xml version="1.0" encoding="UTF-8"?>
    <library>
        <book isbn="123-456">
            <title>Secure Programming</title>
            <author>Security Expert</author>
            <year>2024</year>
        </book>
        <book isbn="789-012">
            <title>XML Security Guide</title>
            <author>Web Security Team</author>
            <year>2023</year>
        </book>
    </library>"""
    
    # Parse with secure ElementTree
    root = SafeET.fromstring(xml_data)
    print(f"    Parsed root element: {root.tag}")
    books = root.findall('book')
    print(f"    Found {len(books)} books")
    
    for book in books:
        isbn = book.get('isbn')
        title = book.find('title').text
        author = book.find('author').text
        year = book.find('year').text
        print(f"      {title} by {author} ({year}) - ISBN: {isbn}")
    
    # 2. SAX parser replacement
    print("  2. SAX parser secure replacement:")
    
    class BookHandler(ContentHandler):
        def __init__(self):
            self.current_element = None
            self.current_book = {}
            self.books = []
            
        def startElement(self, name, attrs):
            self.current_element = name
            if name == 'book':
                self.current_book = {'isbn': attrs.get('isbn', '')}
                
        def endElement(self, name):
            if name == 'book':
                self.books.append(self.current_book.copy())
                self.current_book = {}
            self.current_element = None
            
        def characters(self, content):
            if self.current_element in ['title', 'author', 'year']:
                self.current_book[self.current_element] = content.strip()
    
    # Create secure SAX parser
    parser = make_parser()
    handler = BookHandler()
    parser.setContentHandler(handler)
    
    # Parse XML string with SAX
    from io import StringIO
    SafeSAX.parse(StringIO(xml_data), handler)
    
    print(f"    SAX parsed {len(handler.books)} books:")
    for book in handler.books:
        print(f"      {book.get('title', 'N/A')} - ISBN: {book.get('isbn', 'N/A')}")
    
    # 3. Module comparison
    print("  3. Security module comparison:")
    
    secure_modules = [
        "defusedxml.ElementTree - Secure ElementTree replacement",
        "defusedxml.minidom - Secure minidom replacement", 
        "defusedxml.sax - Secure SAX parser replacement",
        "defusedxml.pulldom - Secure pulldom replacement",
        "defusedxml.xmlrpc - Secure XML-RPC replacement"
    ]
    
    for module in secure_modules:
        print(f"    • {module}")
    
    # 4. Feature comparison
    print("  4. Security features enabled:")
    
    features = [
        "Entity processing disabled by default",
        "External DTD loading blocked",
        "Network access prevention",
        "Recursive entity detection",
        "Memory exhaustion protection",
        "Parser creation with secure defaults"
    ]
    
    for feature in features:
        print(f"    ✓ {feature}")
    
    print("✓ Secure XML module replacements demonstration completed")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: defusedxml provides secure drop-in replacements for XML modules")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"defusedxml example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: defusedxml functionality test 4

In [ ]:
import defusedxml.ElementTree as SafeET
import defusedxml
from xml.etree import ElementTree as StandardET

print("Example 4: Real-world XML Security Applications")

try:
    # Real-world XML security scenarios
    print("✓ Real-world XML security applications:")
    
    # 1. Configuration file parsing
    print("  1. Secure configuration file parsing:")
    
    config_xml = """<?xml version="1.0" encoding="UTF-8"?>
    <configuration>
        <database>
            <host>localhost</host>
            <port>5432</port>
            <name>myapp</name>
            <pool_size>10</pool_size>
        </database>
        <security>
            <enable_ssl>true</enable_ssl>
            <session_timeout>3600</session_timeout>
            <max_login_attempts>5</max_login_attempts>
        </security>
        <features>
            <feature name="caching" enabled="true"/>
            <feature name="logging" enabled="true"/>
            <feature name="monitoring" enabled="false"/>
        </features>
    </configuration>"""
    
    # Parse configuration safely
    config_root = SafeET.fromstring(config_xml)
    print(f"    Configuration loaded from: {config_root.tag}")
    
    # Extract database settings
    db_config = config_root.find('database')
    if db_config is not None:
        host = db_config.find('host').text
        port = int(db_config.find('port').text)
        name = db_config.find('name').text
        pool_size = int(db_config.find('pool_size').text)
        print(f"    Database: {name} at {host}:{port} (pool: {pool_size})")
    
    # Extract security settings
    security_config = config_root.find('security')
    if security_config is not None:
        ssl_enabled = security_config.find('enable_ssl').text.lower() == 'true'
        timeout = int(security_config.find('session_timeout').text)
        max_attempts = int(security_config.find('max_login_attempts').text)
        print(f"    Security: SSL={ssl_enabled}, timeout={timeout}s, max_attempts={max_attempts}")
    
    # Extract feature flags
    features = config_root.findall('.//feature')
    enabled_features = [f.get('name') for f in features if f.get('enabled') == 'true']
    print(f"    Enabled features: {', '.join(enabled_features)}")
    
    # 2. API response parsing
    print("  2. Secure API response parsing:")
    
    api_response = """<?xml version="1.0" encoding="UTF-8"?>
    <response status="success" timestamp="2024-01-15T10:30:00Z">
        <data>
            <user id="12345">
                <username>john_doe</username>
                <email>john@example.com</email>
                <profile>
                    <first_name>John</first_name>
                    <last_name>Doe</last_name>
                    <role>admin</role>
                </profile>
                <permissions>
                    <permission>read</permission>
                    <permission>write</permission>
                    <permission>delete</permission>
                </permissions>
            </user>
        </data>
        <metadata>
            <page>1</page>
            <total_pages>1</total_pages>
            <total_records>1</total_records>
        </metadata>
    </response>"""
    
    # Parse API response safely
    response_root = SafeET.fromstring(api_response)
    status = response_root.get('status')
    timestamp = response_root.get('timestamp')
    print(f"    API Response: {status} at {timestamp}")
    
    # Extract user data
    user = response_root.find('.//user')
    if user is not None:
        user_id = user.get('id')
        username = user.find('username').text
        email = user.find('email').text
        role = user.find('.//role').text
        print(f"    User: {username} ({email}) - ID: {user_id}, Role: {role}")
        
        # Extract permissions
        permissions = [p.text for p in user.findall('.//permission')]
        print(f"    Permissions: {', '.join(permissions)}")
    
    # Extract metadata
    metadata = response_root.find('metadata')
    if metadata is not None:
        page = metadata.find('page').text
        total = metadata.find('total_records').text
        print(f"    Metadata: Page {page}, Total records: {total}")
    
    # 3. Data export/import security
    print("  3. Secure data export/import:")
    
    # Simulate secure data export
    export_data = {
        'users': [
            {'id': 1, 'name': 'Alice', 'department': 'Engineering'},
            {'id': 2, 'name': 'Bob', 'department': 'Marketing'},
            {'id': 3, 'name': 'Carol', 'department': 'Sales'}
        ],
        'export_info': {
            'timestamp': '2024-01-15T10:30:00Z',
            'version': '1.0',
            'format': 'xml'
        }
    }
    
    # Create XML export securely (using xml.etree.ElementTree for creation, then parse with defusedxml)
    from xml.etree.ElementTree import Element, SubElement, tostring

    # Create XML structure using standard ElementTree
    root = Element('export')

    # Add metadata
    info_elem = SubElement(root, 'info')
    for key, value in export_data['export_info'].items():
        elem = SubElement(info_elem, key)
        elem.text = str(value)

    # Add users
    users_elem = SubElement(root, 'users')
    for user_data in export_data['users']:
        user_elem = SubElement(users_elem, 'user')
        user_elem.set('id', str(user_data['id']))

        name_elem = SubElement(user_elem, 'name')
        name_elem.text = user_data['name']

        dept_elem = SubElement(user_elem, 'department')
        dept_elem.text = user_data['department']

    # Convert to string safely
    xml_string = tostring(root, encoding='unicode')
    print(f"    Export created: {len(xml_string)} characters")
    print(f"    Export preview: {xml_string[:100]}...")
    
    # Parse the exported XML back with defusedxml (round-trip test)
    imported_root = SafeET.fromstring(xml_string)
    imported_users = imported_root.findall('.//user')
    print(f"    Import verification: {len(imported_users)} users recovered")
    
    # Demonstrate secure parsing of the created XML
    print("  4. Secure parsing verification:")
    for user in imported_users:
        user_id = user.get('id')
        name = user.find('name').text
        dept = user.find('department').text
        print(f"    Verified user {user_id}: {name} ({dept})")
    
    print("✓ Real-world XML security applications completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: defusedxml enables secure XML processing in real applications")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"defusedxml example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: defusedxml functionality test 5

In [ ]:
import defusedxml
import defusedxml.ElementTree as ET

print("Example 5: defusedxml Package Capabilities and Security Summary")

try:
    # Comprehensive package capabilities demonstration
    print("✓ defusedxml package capabilities summary:")
    
    # 1. Package information and modules
    print("  1. Package overview:")
    print(f"    Main module: {defusedxml.__name__}")
    
    # Available secure modules
    available_modules = []
    secure_replacements = [
        'ElementTree', 'minidom', 'sax', 'pulldom', 'xmlrpc'
    ]
    
    for module_name in secure_replacements:
        try:
            module = getattr(defusedxml, module_name, None)
            if module:
                available_modules.append(f"defusedxml.{module_name}")
        except:
            pass
            raise
    
    print(f"    Available secure modules: {len(available_modules)}")
    for module in available_modules:
        print(f"      • {module}")
    
    # 2. Security features summary
    print("  2. Security features provided:")
    
    security_features = [
        "XML bomb (billion laughs) attack prevention",
        "XXE (XML External Entity) attack blocking",
        "DTD (Document Type Definition) processing restrictions", 
        "External entity reference blocking",
        "Recursive entity expansion detection",
        "Memory exhaustion protection",
        "Network access prevention for DTDs",
        "Entity processing limits and controls",
        "Parser configuration with secure defaults",
        "Drop-in replacements for standard XML modules"
    ]
    
    for i, feature in enumerate(security_features, 1):
        print(f"    {i:2d}. ✓ {feature}")
    
    # 3. Use cases and applications
    print("  3. Common use cases:")
    
    use_cases = [
        "Web application XML parsing",
        "API response processing", 
        "Configuration file reading",
        "Data import/export operations",
        "SOAP and XML-RPC services",
        "RSS/Atom feed processing",
        "Document processing workflows",
        "Secure XML transformation",
        "Multi-tenant application XML handling",
        "Enterprise security compliance"
    ]
    
    for use_case in use_cases:
        print(f"      • {use_case}")
    
    # 4. Performance and compatibility
    print("  4. Performance characteristics:")
    
    # Simple performance demonstration
    sample_xml = """<?xml version="1.0"?>
    <test>
        <item id="1">Sample data 1</item>
        <item id="2">Sample data 2</item>
        <item id="3">Sample data 3</item>
    </test>"""
    
    import time
    
    # Time the parsing operation
    start_time = time.time()
    for i in range(100):
        root = ET.fromstring(sample_xml)
        items = root.findall('item')
    parse_time = time.time() - start_time
    
    print(f"    Parsing performance: {parse_time*1000:.2f}ms for 100 operations")
    print(f"    Security overhead: Minimal impact on parsing speed")
    print(f"    Memory usage: Efficient with built-in protection")
    print(f"    Compatibility: Drop-in replacement for standard XML libraries")
    
    # 5. Best practices and recommendations
    print("  5. Implementation best practices:")
    
    best_practices = [
        "Always use defusedxml instead of standard xml modules",
        "Validate XML structure before processing sensitive data",
        "Implement additional input validation for untrusted XML",
        "Monitor and log XML processing for security auditing",
        "Configure custom limits for specific application needs",
        "Test with various XML attack vectors during development",
        "Keep defusedxml updated to latest security patches",
        "Combine with other security measures (input sanitization)",
        "Document XML processing security measures for compliance"
    ]
    
    for practice in best_practices:
        print(f"      • {practice}")
    
    # 6. Final demonstration
    print("  6. Package verification:")
    
    verification_xml = """<?xml version="1.0"?>
    <verification>
        <status>secure</status>
        <module>defusedxml</module>
        <features_enabled>true</features_enabled>
    </verification>"""
    
    # Verify parsing works correctly
    verify_root = ET.fromstring(verification_xml)
    status = verify_root.find('status').text
    module = verify_root.find('module').text
    features = verify_root.find('features_enabled').text
    
    print(f"    Verification status: {status}")
    print(f"    Module confirmed: {module}")
    print(f"    Security features: {features}")
    
    print("✓ defusedxml package capabilities summary completed successfully")
    print("  → Ready for secure XML processing in production environments")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: Install defusedxml for secure XML processing: pip install defusedxml")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"defusedxml example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")